# Lab 07 – Product Recommendation System (Random Forest Candidate Scoring)
**Course:** MDI3003 – Advanced Predictive Analytics
**Student:** Geetha Priya S | **Roll No:** 23MID0021

Supervised recommendation pipeline on a synthetic UCI Online Retail dataset. A Random Forest classifier scores user-item candidate pairs using 17 engineered features to produce personalised top-K product recommendations.

In [ ]:
# !pip install pandas numpy scikit-learn matplotlib joblib

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (precision_score, recall_score, roc_auc_score,
                              average_precision_score)
import joblib
warnings.filterwarnings('ignore')
np.random.seed(42)
print("Libraries loaded.")

## 1. Load Dataset & Artefacts

In [ ]:
BASE      = r'c:\Users\I768951\OneDrive - SAP SE\SAP\Exam'
OUT       = os.path.join(BASE, 'lab07_outputs')
ARTIFACTS = os.path.join(BASE, 'artifacts')
MDL       = os.path.join(BASE, 'models')

df = pd.read_csv(os.path.join(OUT, 'dataset.csv'), parse_dates=['InvoiceDate'])
mf = json.load(open(os.path.join(ARTIFACTS, 'split_manifest.json')))
cp = json.load(open(os.path.join(ARTIFACTS, 'candidate_policy.json')))

print(f"Raw rows     : {len(df):,}")
print(f"Customers    : {df['CustomerID'].nunique():,}")
print(f"Items        : {df['StockCode'].nunique():,}")
print(f"Split t1     : {mf['t1']}")
print(f"Split t2     : {mf['t2']}")
print(f"Catalog size : {cp['catalog_size']}")
print(f"Cand. recall : {cp['candidate_recall_pct']:.2f}%")

## 2. Data Cleaning & Chronological Split

In [ ]:
# Remove cancellations and invalid rows
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df = df.dropna(subset=['CustomerID'])
df = df[(df['Quantity']>0) & (df['UnitPrice']>0)].copy()
df['revenue'] = df['Quantity'] * df['UnitPrice']

t1 = pd.Timestamp(mf['t1'])
t2 = pd.Timestamp(mf['t2'])

train = df[df['InvoiceDate'] <= t1]
val   = df[(df['InvoiceDate'] > t1) & (df['InvoiceDate'] <= t2)]
test  = df[df['InvoiceDate'] > t2]

print(f"Clean rows : {len(df):,}")
print(f"Train      : {len(train):,}  ({train['InvoiceDate'].min().date()} → {train['InvoiceDate'].max().date()})")
print(f"Val        : {len(val):,}")
print(f"Test       : {len(test):,}")

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Monthly transaction volume
monthly = train.copy()
monthly['month'] = monthly['InvoiceDate'].dt.to_period('M')
mv = monthly.groupby('month')['InvoiceNo'].nunique()
axes[0].bar(range(len(mv)), mv.values, color='#2E86AB')
axes[0].set_xticks(range(len(mv))); axes[0].set_xticklabels([str(m) for m in mv.index], rotation=45, fontsize=7)
axes[0].set_title('Monthly Transactions (Train)'); axes[0].set_ylabel('Invoice Count')

# Top 15 items by buyers
top_items = train.groupby('StockCode')['CustomerID'].nunique().nlargest(15)
axes[1].barh(range(15), top_items.values[::-1], color='#A23B72')
axes[1].set_yticks(range(15)); axes[1].set_yticklabels([str(x) for x in top_items.index[::-1]], fontsize=7)
axes[1].set_title('Top 15 Items by Unique Buyers'); axes[1].set_xlabel('Buyers')

# Customer purchase frequency
freq = train.groupby('CustomerID')['StockCode'].nunique()
axes[2].hist(freq, bins=30, color='#F18F01', edgecolor='white')
axes[2].set_title('Customer Purchase Variety'); axes[2].set_xlabel('Unique Items Bought')
axes[2].set_ylabel('Customers')

plt.tight_layout()
plt.savefig(os.path.join(BASE, 'lab07_nb_eda.png'), dpi=100, bbox_inches='tight')
plt.show()

## 4. Feature Engineering (17 Features)

In [ ]:
ref_date = pd.Timestamp(mf['train_end'])
CATS = sorted(train['Category'].unique()); cat_map = {c:i for i,c in enumerate(CATS)}

# Customer features (6)
cf = train.groupby('CustomerID').agg(
    recency_days=('InvoiceDate', lambda x: (ref_date-x.max()).days),
    cust_txns   =('InvoiceNo',   'nunique'),
    cust_items  =('StockCode',   'count'),
    cust_spend  =('revenue',     'sum'),
    cust_uniq   =('StockCode',   'nunique'),
).reset_index()
cf['pref_cat_enc'] = train.groupby('CustomerID')['Category'].agg(lambda x: x.mode()[0]).map(cat_map).values

# Item features (6)
itf = train.groupby('StockCode').agg(
    item_txns    =('InvoiceNo',   'nunique'),
    item_buyers  =('CustomerID',  'nunique'),
    item_avg_price=('UnitPrice',  'mean'),
    item_recency_days=('InvoiceDate', lambda x: (ref_date-x.max()).days),
    item_cat_enc =('Category',    lambda x: cat_map.get(x.mode()[0], 0)),
).reset_index()
item_pop = train.groupby('StockCode')['CustomerID'].nunique().rank(pct=True)
itf['item_pop_rank'] = itf['StockCode'].map(item_pop)

print(f"Customer features: {cf.shape}")
print(f"Item features    : {itf.shape}")
print("\nCustomer feature sample:")
print(cf.head(3).to_string(index=False))

## 5. Load Saved Random Forest & Recommendations

In [ ]:
# Load the pre-trained model (avoids 5+ min retraining in notebook)
rf_path = os.path.join(MDL, 'random_forest.joblib')
rf = joblib.load(rf_path)
print(f"Loaded: {rf_path}")
print(f"Trees : {rf.n_estimators}  |  Features: {rf.n_features_in_}")

# Load saved recommendations
recs = pd.read_csv(os.path.join(OUT, '23MID0021_Lab07_Recommendations.csv'))
print(f"\nRecommendations: {len(recs):,} rows for {recs['CustomerID'].nunique():,} customers")
recs.head(10)

## 6. Test Set Evaluation Metrics

In [ ]:
metrics_df = pd.read_csv(os.path.join(OUT, '23MID0021_Lab07_Ranking_Metrics.csv'))
print(metrics_df.to_string(index=False))

In [ ]:
# Plot Precision@K and Recall@K
rf_m = metrics_df[metrics_df['Model']=='RandomForest']
pop_m = metrics_df[metrics_df['Model']=='Popularity']
cf_m  = metrics_df[metrics_df['Model']=='Item-CF']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, title in zip(axes, ['Precision','Recall','HitRate'], ['Precision@K','Recall@K','HitRate@K']):
    for mdf, name, color in [(pop_m,'Popularity','#F18F01'),(cf_m,'Item-CF','#A23B72'),(rf_m,'RandomForest','#2E86AB')]:
        ax.plot(mdf['K'], mdf[col], 'o-', color=color, label=name, lw=2)
    ax.set_xlabel('K'); ax.set_ylabel(col); ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(BASE, 'lab07_nb_metrics.png'), dpi=100, bbox_inches='tight')
plt.show()

## 7. Top-10 Recommendations – Sample Customers

In [ ]:
sample_customers = recs['CustomerID'].drop_duplicates().head(5).values

for cid in sample_customers:
    top10 = recs[recs['CustomerID']==cid].nsmallest(10, 'rank')[['rank','StockCode','Description','score']]
    print(f"\nCustomer {cid} – Top-10 Recommendations:")
    print(top10.to_string(index=False))

## 8. Feature Importance

In [ ]:
FEAT_NAMES = [
    'recency_days','cust_txns','cust_items','cust_spend','cust_uniq','pref_cat_enc',
    'item_txns','item_buyers','item_avg_price','item_recency_days','item_cat_enc','item_pop_rank',
    'pair_purchases','pair_qty','pair_spend','pair_days_since','cat_match'
]

importances = rf.feature_importances_
idx = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(idx)), importances[idx], color='#2E86AB')
ax.set_xticks(range(len(idx)))
ax.set_xticklabels([FEAT_NAMES[i] for i in idx], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Mean Decrease in Impurity')
ax.set_title('Random Forest Feature Importances', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'lab07_nb_importance.png'), dpi=100, bbox_inches='tight')
plt.show()

## 9. Save / Reload Model Verification

In [ ]:
# Verify saved model loads and predicts correctly
rf2 = joblib.load(rf_path)
x_sample = np.array([[15, 8, 42, 320.5, 12, 2,
                       55, 30, 4.25, 10, 2, 0.15,
                       2, 5, 8.50, 7, 1]])
prob = rf2.predict_proba(x_sample)[0][1]
print(f"Model reloaded successfully.")
print(f"P(purchase) for sample feature vector: {prob:.4f}")

## 10. Summary

In [ ]:
print("=" * 60)
print("LAB 07 RESULTS SUMMARY")
print("=" * 60)
rf_k10 = metrics_df[(metrics_df['Model']=='RandomForest') & (metrics_df['K']==10)].iloc[0]
pop_k10= metrics_df[(metrics_df['Model']=='Popularity')   & (metrics_df['K']==10)].iloc[0]
cf_k10 = metrics_df[(metrics_df['Model']=='Item-CF')      & (metrics_df['K']==10)].iloc[0]

print(f"{'Model':<20} {'P@10':<8} {'R@10':<8} {'HR@10':<8} {'NDCG@10'}")
for row, name in [(pop_k10,'Popularity'),(cf_k10,'Item-CF'),(rf_k10,'RandomForest')]:
    print(f"{name:<20} {row['Precision']:.4f}   {row['Recall']:.4f}   {row['HitRate']:.4f}   {row['NDCG']:.4f}")
print()
print(f"RF is {rf_k10['NDCG']/pop_k10['NDCG']:.1f}x better NDCG@10 vs Popularity baseline")